# Train And Test

Here, we will train and test our dataset with different model and find the importance of different feature on our Churn rate. 

### Importing all necessary libraries

In [ ]:
import pandas as pd
from pipeline_utils import get_features,get_preprocessor,load_and_split_data
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import f1_score, recall_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

### Using pipelines to get data, encode them and split it for training and testing purpose

In [ ]:
le=LabelEncoder()

In [ ]:
# Load dataset and separate features (X) from the target (Y)
X, Y = load_and_split_data('telco_churn.csv')

# Encode target labels
Y = le.fit_transform(Y)

# Get column names grouped by their type
num_cols, ord_cols, nom_cols = get_features()

# Set up the preprocessing pipeline
preprocessor = get_preprocessor(num_cols, ord_cols, nom_cols)

# Standard 80/20 split for training and testing
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=42)

# Dictionary to keep track of CV scores for model.
model_scores = {}


### Traning and testing them using differnet models

#### AdaBoost Classifier

In [ ]:
# AdaBoost Classifier

# Creating the model and fitting it into the pipeline

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', AdaBoostClassifier(n_estimators=100))
])

model.fit(X_train,Y_train)


# Testing the AdaBoost Classifier

Y_predict=model.predict(X_test)
f1 = f1_score(Y_test, Y_predict)
recall = recall_score(Y_test, Y_predict)
print(f"Test F1-Score: {f1:.4f}")
print(f"Test Recall: {recall:.4f}")


# Hyperparameter tuning for AdaBoostClassifier

param_grid={'classifier__n_estimators':[100,150,200]}

grid=GridSearchCV(model,param_grid,cv=5, scoring='f1')
grid.fit(X_train, Y_train)
print(f"Best parameters: {grid.best_params_}")
print(f"Best cross-validation score: {grid.best_score_}")


# Testing the tuned AdaBoost Classifier

Y_predict=grid.predict(X_test)
f1 = f1_score(Y_test, Y_predict)
recall = recall_score(Y_test, Y_predict)
print(f"Test F1-Score_tuned: {f1:.4f}")
print(f"Test Recall_tuned: {recall:.4f}")

model_scores['AdaBoost'] = {'F1-Score': f1, 'Recall': recall}

#### XGB Classifier

In [ ]:
# XGB Classifier

# The ratio of stayers to churners to balance XGBoost

scale_pos_weight = len(Y_train[Y_train == 0]) / len(Y_train[Y_train == 1])


# Creating the model and fitting it into the pipeline

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(n_estimators=100, scale_pos_weight=scale_pos_weight))
])

model.fit(X_train,Y_train)


# Testing the XGB Classifier

Y_predict=model.predict(X_test)
f1 = f1_score(Y_test, Y_predict)
recall = recall_score(Y_test, Y_predict)
print(f"Test F1-Score: {f1:.4f}")
print(f"Test Recall: {recall:.4f}")


# Hyperparameter tuning for XGBClassifier

param_grid={'classifier__n_estimators':[50,100]}

grid=GridSearchCV(model,param_grid,cv=5, scoring='f1')
grid.fit(X_train, Y_train)
print(f"Best parameters: {grid.best_params_}")
print(f"Best cross-validation score: {grid.best_score_}")


# Testing the tuned XGB Classifier

Y_predict=grid.predict(X_test)
f1 = f1_score(Y_test, Y_predict)
recall = recall_score(Y_test, Y_predict)
print(f"Test F1-Score_tuned: {f1:.4f}")
print(f"Test Recall_tuned: {recall:.4f}")

model_scores['XGBoost'] = {'F1-Score': f1, 'Recall': recall}

#### CatBoost Classifier

In [ ]:
# CatBoost Classifier

# Creating the model and fitting it into the pipeline

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', CatBoostClassifier(n_estimators=100, auto_class_weights='Balanced', verbose=False))
])

model.fit(X_train,Y_train)


# Testing the CatBoost Classifier

Y_predict=model.predict(X_test)
f1 = f1_score(Y_test, Y_predict)
recall = recall_score(Y_test, Y_predict)
print(f"Test F1-Score: {f1:.4f}")
print(f"Test Recall: {recall:.4f}")


# Hyperparameter tuning for CatBoostClassifier

param_grid={'classifier__n_estimators':[200,250,300]}

grid=GridSearchCV(model,param_grid,cv=5, scoring='f1')
grid.fit(X_train, Y_train)
print(f"Best parameters: {grid.best_params_}")
print(f"Best cross-validation score: {grid.best_score_}")


# Testing the tuned CatBoost Classifier

Y_predict=grid.predict(X_test)
f1 = f1_score(Y_test, Y_predict)
recall = recall_score(Y_test, Y_predict)
print(f"Test F1-Score_tuned: {f1:.4f}")
print(f"Test Recall_tuned: {recall:.4f}")

model_scores['CatBoost'] = {'F1-Score': f1, 'Recall': recall}




#### Kneighbours Classifier

In [ ]:
# KNeighbors Classifier

# Creating the model and fitting it into the pipeline

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier(n_neighbors=5))
])

model.fit(X_train,Y_train)


# Testing the KNeighbors Classifier

Y_predict=model.predict(X_test)
f1 = f1_score(Y_test, Y_predict)
recall = recall_score(Y_test, Y_predict)
print(f"Test F1-Score: {f1:.4f}")
print(f"Test Recall: {recall:.4f}")


# Hyperparameter tuning for KNeighborsClassifier

param_grid={'classifier__n_neighbors':[7,9,11]}

grid=GridSearchCV(model,param_grid,cv=5, scoring='f1')
grid.fit(X_train, Y_train)
print(f"Best parameters: {grid.best_params_}")
print(f"Best cross-validation score: {grid.best_score_}")


# Testing the tuned KNeighbors Classifier

Y_predict=grid.predict(X_test)
f1 = f1_score(Y_test, Y_predict)
recall = recall_score(Y_test, Y_predict)
print(f"Test F1-Score_tuned: {f1:.4f}")
print(f"Test Recall_tuned: {recall:.4f}")

model_scores['KNeighbors'] = {'F1-Score': f1, 'Recall': recall}

#### Random Forest Classifier

In [ ]:
# Random Forest Classifier

# Creating the model and fitting it into the pipeline

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(max_depth=5, class_weight='balanced', random_state=42))
])

model.fit(X_train,Y_train)


# Testing the Random Forest Classifier


Y_predict=model.predict(X_test)
f1 = f1_score(Y_test, Y_predict)
recall = recall_score(Y_test, Y_predict)
print(f"Test F1-Score: {f1:.4f}")
print(f"Test Recall: {recall:.4f}")


# Hyperparameter tuning for RandomForestClassifier

param_grid={'classifier__n_estimators':[150,200,250], 'classifier__max_depth':[5,6,7]}

grid=GridSearchCV(model,param_grid,cv=5, scoring='f1')
grid.fit(X_train, Y_train)
print(f"Best parameters: {grid.best_params_}")
print(f"Best cross-validation score: {grid.best_score_}")


# Testing the tuned Random Forest Classifier

Y_predict=grid.predict(X_test)
f1 = f1_score(Y_test, Y_predict)
recall = recall_score(Y_test, Y_predict)
print(f"Test F1-Score_tuned: {f1:.4f}")
print(f"Test Recall_tuned: {recall:.4f}")


model_scores['Random Forest'] = {'F1-Score': f1, 'Recall': recall}

### Comparing the result of different models:

In [ ]:
# 1. Convert the nested dictionary into a DataFrame
scores_df = pd.DataFrame.from_dict(model_scores, orient='index').reset_index()
scores_df.rename(columns={'index': 'Model'}, inplace=True)

# Sort by F1-Score so the best overall model is at the top
scores_df = scores_df.sort_values(by='F1-Score', ascending=False)

scores_df.to_csv('model_scores.csv', index=False)

# Display the text table
print("Final Model Rankings (Test Data):")
display(scores_df)

# 2. Reshape the DataFrame for Seaborn (Melting)
scores_melted = scores_df.melt(id_vars='Model', var_name='Metric', value_name='Score')

# 3. Plot the Grouped Bar Chart
plt.figure(figsize=(12, 7))
ax = sns.barplot(data=scores_melted, x='Score', y='Model', hue='Metric', palette='Set2')
plt.title('Final Showdown: F1-Score vs Recall on Unseen Test Data')
plt.xlim(0, 1) # Set x-axis from 0 to 1

# Add the exact numbers to the bars
for p in ax.patches:
    width = p.get_width()
    if width > 0: # Only add text if there's a bar
        plt.text(width + 0.01, p.get_y() + p.get_height() / 2, f'{width:.3f}', va='center')

plt.tight_layout()
plt.show()

### Understanding the Metrics:
+ Recall (Primary Metric): In customer churn forecasting, False Negatives carry the highest financial penalty. Failing to identify a customer who is about to leave results in a direct loss of revenue and Customer Lifetime Value (LTV). Prioritizing Recall ensures the model maximizes the detection of true churn risks.
+ F1-Score (Mathematical Safeguard): While high Recall is crucial, relying on it exclusively creates a moral hazard: a baseline model could achieve a perfect 1.0 Recall by universally classifying every user as a churner. The F1-Score (the harmonic mean of Precision and Recall) serves as a critical sanity check, ensuring the model remains precise enough to prevent the business from wasting retention budgets on perfectly stable, loyal accounts.

### Understanding the Model:
+ Comparative Benchmarking: After evaluating five distinct machine learning architectures integrated with automated preprocessing pipelines and rigorous hyperparameter tuning, the Tuned Random Forest Classifier emerged as the clear champion. It outpaced gradient boosting variations (CatBoost, XGBoost) and simpler baseline models by establishing the strongest simultaneous boundary for both core metrics.
+ Model Performance: On entirely unseen test data, the optimized Random Forest achieved a dominant Recall of 0.800 alongside a highly stable F1-Score of 0.673.
+ Operational Business Impact: By capturing 80% of actual churn events (8 out of 10 departing customers), this predictive pipeline shifts the customer success team from a reactive posture to a highly strategic, proactive window—allowing the enterprise to deploy targeted intervention plays before a clean break occurs.

### Identifying important features

In [ ]:
# The importances
model.fit(X_train, Y_train)
importances = model.named_steps['classifier'].feature_importances_
feature_names = model.named_steps['preprocessor'].named_steps['column_transformations'].get_feature_names_out()

# Create a dataframe for visualization
feature_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_imp_df = feature_imp_df.sort_values(by='Importance', ascending=False)

# Plotting
plt.figure(figsize=(10, 12))
sns.barplot(data=feature_imp_df, x='Importance', y='Feature')
plt.title('Random Forest Feature Importance')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.show()

### Feature Interpretability & Strategic Business Insights
Extracting the feature importances from the Random Forest Classifier provides global model interpretability, shifting the project from purely predictive to highly prescriptive. The distribution reveals that structural, lifecycle, and product-engagement factors heavily dominate the model's decision-making process, while demographic traits possess zero predictive signal.

+ Contract Type and Lifecycle Tenure (Primary Predictors): Contract and tenure emerged as the most influential variables driving the model. The heavy weight placed on Contract indicates that short-term billing cycles (such as month-to-month arrangements) are volatile structural risks.  
    * Strategic Recommendation: The business should prioritize migrating month-to-month users into long-term agreements. Introducing tiered financial incentives or exclusive product additions for extended commitments will stabilize customer lifecycles and directly lower churn.
+ The "Product Stickiness" Gap (Value-Added Services): The strong predictive weight of OnlineSecurity_No and TechSupport_No highlights a critical risk factor. The absence of these core utility features strongly flags a customer's likelihood to leave.  
    * Strategic Recommendation: Customer retention efforts should focus on cross-selling or automatically bundling Online Security and Tech Support features into baseline subscription packages. Increasing adoption rates for these specific add-ons acts as an operational anchor, making the service more vital to the user.
+ Financial and Technical Friction Points: Beyond contract structure, high financial indicators (TotalCharges, MonthlyCharges) and specific plan options (InternetService_Fiber optic, PaymentMethod_Electronic check) show notable predictive value. This suggests that users experiencing high monthly out-of-pocket costs, combined with manual billing friction (like non-automated electronic checks), present an elevated risk profile. 
    * Strategic Recommendation: Target these high-value segments with proactive, automated loyalty discounts and introduce seamless, automated payment incentives (such as credit card auto-pay bonuses) to eliminate transactional friction.
+ Inconsequential Attributes (Demographic Neutrality): The model explicitly demonstrates that demographic features—specifically gender_Male and gender_Female—hold negligible importance.
    * Operational Takeaway: Marketing and customer success campaigns should remain completely blind to gender demographics. Instead, resource allocation must prioritize user behavioral interactions, contract structures, and product usage patterns to maximize retention ROI.

### Saving the best model (Random Forest) to disk

In [ ]:
best_model = grid.best_estimator_

with open('telco_churn_pipeline.pkl', 'wb') as f:
    pickle.dump(best_model, f)

# THE END OF TRAIN_TEST NOTEBOOK